# Distillation System Identification

This notebook generates the canonical distillation linear-model assets in `Distillation/Data` and keeps Aspen/data controls at the top.

In [ ]:
from pathlib import Path
import os
import pickle

from utils.notebook_setup import prepare_distillation_notebook_env, print_notebook_summary

RUN_NEW_EXPERIMENTS = False
USE_RHP_ZERO = True
SHOW_FOPDT_PLOTS = True
SHOW_VALIDATION_PLOTS = True

ASPEN_PRESET = "default"
ASPEN_PATH_OVERRIDE = None
SNAPS_PATH_OVERRIDE = None
ASPEN_ROOT_OVERRIDE = None
DISTILLATION_VISIBLE = True

DISTILLATION_DATA_DIR_OVERRIDE = None
DISTILLATION_RESULTS_DIR_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(
    run_mode="nominal",
    disturbance_profile="none",
    family="system_id",
    aspen_preset=ASPEN_PRESET,
    dyn_path_override=ASPEN_PATH_OVERRIDE,
    snaps_path_override=SNAPS_PATH_OVERRIDE,
    aspen_root_override=ASPEN_ROOT_OVERRIDE,
    data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE,
    results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)
CARRY_FORWARD_MIN_MAX = DATA_DIR / "min_max_states.pickle"

print_notebook_summary(
    "Distillation system-identification configuration",
    {
        "Distillation data dir": DATA_DIR,
        "Distillation results": RESULT_DIR,
        "Aspen source": ASPEN_SOURCE,
        "Aspen dynf": DYN_PATH,
        "Aspen snaps": SNAPS_PATH,
        "Run new tests": RUN_NEW_EXPERIMENTS,
    },
)

In [ ]:
import numpy as np
import pandas as pd

from systems.distillation.config import (
    DELTA_T_HOURS,
    DISTILLATION_NOMINAL_CONDITIONS,
    DISTILLATION_SS_INPUTS_SYSTEM_ID,
    )
from systems.distillation.data_io import copy_legacy_distillation_data, resolve_distillation_data_dir
from systems.distillation.plant import DistillationColumnAspen, build_distillation_system
from systems.distillation.system_id import (
    apply_deviation_form_scaled,
    build_delay_list_hours,
    extract_fopdt_2863,
    plot_state_space_validation,
    run_distillation_experiment,
    save_canonical_system_identification,
    scaling_min_max_factors,
    state_space_form_using_matlab,
)


In [ ]:
DATA_DIR = resolve_distillation_data_dir(REPO_ROOT)
CARRY_FORWARD_MIN_MAX = DATA_DIR / "min_max_states.pickle"
DYN_PATH_DEFAULT, SNAPS_PATH_DEFAULT = default_plant_paths("system_id", "none")
DYN_PATH = Path(DISTILLATION_DYN_PATH).expanduser() if DISTILLATION_DYN_PATH else DYN_PATH_DEFAULT
SNAPS_PATH = Path(DISTILLATION_SNAPS_PATH).expanduser() if DISTILLATION_SNAPS_PATH else SNAPS_PATH_DEFAULT

nominal_conditions = DISTILLATION_NOMINAL_CONDITIONS.copy()
ss_inputs = DISTILLATION_SS_INPUTS_SYSTEM_ID.copy()

print(f"Canonical distillation data dir: {DATA_DIR}")
print(f"Using Aspen dynf: {DYN_PATH}")


In [ ]:
steady_states = {"ss_inputs": ss_inputs.copy(), "y_ss": None}

if RUN_NEW_EXPERIMENTS:
    system = build_distillation_system(
        path=DYN_PATH,
        ss_inputs=ss_inputs,
        initialization_point=nominal_conditions,
        delta_t=DELTA_T_HOURS,
        visible=DISTILLATION_VISIBLE,
    )
    try:
        steady_states = {
            "ss_inputs": np.asarray(system.ss_inputs, float).copy(),
            "y_ss": np.asarray(system.y_ss, float).copy(),
        }
    finally:
        try:
            system.close(SNAPS_PATH)
        except Exception:
            pass

    run_distillation_experiment(
        system_cls=DistillationColumnAspen,
        path=DYN_PATH,
        ss_inputs=steady_states["ss_inputs"],
        nominal_conditions=nominal_conditions,
        delta_t=DELTA_T_HOURS,
        step_value=-40000.0,
        step_channel=0,
        save_filename="Reflux.csv",
        data_dir=DATA_DIR,
    )
    run_distillation_experiment(
        system_cls=DistillationColumnAspen,
        path=DYN_PATH,
        ss_inputs=steady_states["ss_inputs"],
        nominal_conditions=nominal_conditions,
        delta_t=DELTA_T_HOURS,
        step_value=-15.0,
        step_channel=1,
        save_filename="Reboiler.csv",
        data_dir=DATA_DIR,
    )
else:
    print("RUN_NEW_EXPERIMENTS=False, using the canonical/archived distillation data already present in Distillation/Data.")

print(steady_states)


In [ ]:
file_paths = {
    "Reflux": DATA_DIR / "Reflux.csv",
    "Reboiler": DATA_DIR / "Reboiler.csv",
}

data_min, data_max = scaling_min_max_factors(file_paths)
scaling_factor = {
    "min": np.asarray(data_min, float),
    "max": np.asarray(data_max, float),
}

with open(DATA_DIR / "system_dict.pickle", "rb") as handle:
    legacy_system_dict = pickle.load(handle)
with open(CARRY_FORWARD_MIN_MAX, "rb") as handle:
    legacy_min_max_states = pickle.load(handle)

if RUN_NEW_EXPERIMENTS:
    if steady_states["y_ss"] is None:
        raise RuntimeError("Steady-state outputs are required when regenerating the distillation identification assets.")
    deviation_dfs = apply_deviation_form_scaled(steady_states, file_paths, data_min, data_max)
else:
    deviation_dfs = {name: pd.read_csv(path) for name, path in file_paths.items()}

print("Scaling min:", scaling_factor["min"])
print("Scaling max:", scaling_factor["max"])


In [ ]:
if RUN_NEW_EXPERIMENTS:
    reflux_dict, reflux_data = extract_fopdt_2863(
        deviation_dfs["Reflux"],
        input_idx=0,
        Ts=DELTA_T_HOURS,
        limit=25,
        pre_win=30,
        post_win=30,
        plot=SHOW_FOPDT_PLOTS,
    )
    reboiler_dict, reboiler_data = extract_fopdt_2863(
        deviation_dfs["Reboiler"],
        input_idx=1,
        Ts=DELTA_T_HOURS,
        limit=500,
        pre_win=30,
        post_win=500,
        plot=SHOW_FOPDT_PLOTS,
    )

    delay_list_hours, delay_samples = build_delay_list_hours(
        reflux_dict,
        reboiler_dict,
        y1_name="Tray24_C2H6",
        y2_name="Tray85_T",
        Ts_hours=DELTA_T_HOURS,
        quantization="ceil",
    )

    A, B, C, D, y_step_sum, y_step, tOut = state_space_form_using_matlab(
        u1_dict=reflux_dict,
        u2_dict=reboiler_dict,
        delay_list_hours=delay_list_hours,
        data_u1=reflux_data,
        data_u2=reboiler_data,
        sampling_time=DELTA_T_HOURS,
        use_rhp_zero=USE_RHP_ZERO,
        return_step=True,
    )

    if SHOW_VALIDATION_PLOTS:
        plot_state_space_validation(tOut, y_step, reflux_data, reboiler_data, Ts=DELTA_T_HOURS)

    system_dict = {"A": A, "B": B, "C": C, "D": D}
else:
    system_dict = legacy_system_dict

system_dict


In [ ]:
save_canonical_system_identification(
    data_dir=DATA_DIR,
    system_dict=system_dict,
    scaling_factor=scaling_factor,
    min_max_states=legacy_min_max_states,
)

print(f"Saved canonical distillation identification assets to {DATA_DIR}")
print(sorted(p.name for p in DATA_DIR.glob("*.pickle")))
